# Top 5 UCB mới — kiểm định độc lập trên 20 seed

Notebook chạy đúng một trong năm cấu hình UCB tốt nhất trên mỗi tài khoản Kaggle.

- Mỗi tài khoản chỉ đổi CONFIG_SLOT thành 1, 2, 3, 4 hoặc 5.
- Mỗi cấu hình huấn luyện độc lập trên 20 seed 45–64; không đọc hoặc gộp kết quả từ tài khoản khác.
- Train trên HPG BAD train chính thức, đánh giá trên HPG BAD test chính thức.
- Giữ mô hình cải tiến từ tim_tham_so.ipynb: reward shaping, Huber loss, weight decay, frozen scaler chỉ fit trên train, joint state–action VAE vae_019, cost network và confidence-weighted risk penalty.
- Checkpoint sau từng seed và tự động resume.
- Biểu đồ train–test theo episode, mean ± std, phân phối 20 seed, generalization gap, scatter train–test và loss convergence hỗ trợ quan sát overfit.

Test được đánh giá theo episode chỉ để chẩn đoán; không early-stop và không cập nhật mô hình bằng dữ liệu test.

In [ ]:
!if [ ! -d SARSA_FinancialRL ]; then git clone https://github.com/kohi-vip/SARSA_FinancialRL.git; else echo 'SARSA_FinancialRL already exists'; fi


In [ ]:
!pip install numpy pandas matplotlib tqdm torch TA-Lib optuna tabulate


In [ ]:
# CODE 1 — Cấu hình tài khoản, top-5 UCB và 20 seed độc lập
from __future__ import annotations
import gc, json, os, random, time, traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

# Trên 5 tài khoản Kaggle, chỉ đổi CONFIG_SLOT lần lượt thành 1, 2, 3, 4, 5.
CONFIG_SLOT = 1
RUN_TRAINING = True
RESUME = True
VALIDATION_SEEDS = tuple(range(45, 65))

TOP5_UCB_CONFIGS = {
    1: {"config_id": "ucb_002", "beta_0": 0.01, "beta_decay": 0.93, "beta_min": 0.005},
    2: {"config_id": "ucb_009", "beta_0": 0.03, "beta_decay": 0.90, "beta_min": 0.010},
    3: {"config_id": "ucb_008", "beta_0": 0.03, "beta_decay": 0.90, "beta_min": 0.005},
    4: {"config_id": "ucb_041", "beta_0": 0.30, "beta_decay": 0.90, "beta_min": 0.010},
    5: {"config_id": "ucb_027", "beta_0": 0.10, "beta_decay": 0.93, "beta_min": 0.010},
}
if CONFIG_SLOT not in TOP5_UCB_CONFIGS:
    raise ValueError("CONFIG_SLOT phải thuộc {1, 2, 3, 4, 5}.")
SELECTED_UCB = dict(TOP5_UCB_CONFIGS[CONFIG_SLOT])

LOCKED_VAE_CONFIG = {
    "vae_latent_dim": 16, "vae_lr": 1e-4, "vae_beta_kl": 5e-2,
    "vae_batch_size": 256, "bootstrap_trajectories": 5,
    "bootstrap_updates": 100, "online_aux_updates": 1,
    "vae_replay_capacity": 50_000,
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA, NN_LR, COST_LR, EPISODES = 0.95, 5e-5, 1e-3, 45
SARSA_ALPHA, Q_WEIGHT_DECAY, BATCH_SIZE = 0.60, 1e-4, 128
LATENT_DIM, BALANCE_INIT, TRANSACTION_FEE = 16, 1_000.0, 0.001
W_RISK, W_STABILITY, ZETA = 0.15, 0.05, 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT, SCALER_SEED = 2.0, 43
CURRENT_RUN_SEED = VALIDATION_SEEDS[0]

RUN_CONFIG = {
    "label": f"Improved UCB-VAE — {SELECTED_UCB['config_id']}",
    "use_cost": True, "robust_loss": True, "weight_decay": Q_WEIGHT_DECAY,
    "reward_shaping": True, "kl_reduction": "sum",
    **LOCKED_VAE_CONFIG, **SELECTED_UCB,
}

def set_seed(seed: int) -> None:
    global CURRENT_RUN_SEED
    CURRENT_RUN_SEED = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError: torch.use_deterministic_algorithms(True)

set_seed(VALIDATION_SEEDS[0])
print({"device": str(DEVICE), "config_slot": CONFIG_SLOT, "selected_ucb": SELECTED_UCB,
       "locked_vae": LOCKED_VAE_CONFIG, "seeds": VALIDATION_SEEDS, "runs": len(VALIDATION_SEEDS)})

In [ ]:
# CODE 2 — Dữ liệu HPG BAD, môi trường cải tiến và frozen scaler chỉ fit trên train
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists(): return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
OUTPUT_DIR = OUTPUT_ROOT / "Top_5_UCB_new" / SELECTED_UCB["config_id"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]

def load_hpg_bad() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing: raise ValueError(f"HPG BAD {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        frame[REQUIRED_COLUMNS[1:]] = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if frame[REQUIRED_COLUMNS[1:]].isna().any().any():
            raise ValueError(f"HPG BAD {label} chứa NaN hoặc giá trị không hợp lệ.")
    if train["time"].max() >= test["time"].min(): raise ValueError("Train/Test chồng lấn thời gian.")
    return train.reset_index(drop=True), test.reset_index(drop=True)

def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray([row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]], dtype=np.float32)

class TradingEnv:
    """Môi trường và reward cải tiến giữ nguyên từ tim_tham_so.ipynb."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2: raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True); self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash); self.reset()
    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash; self.portfolio_history = [self.initial_cash]; self.var_targets = []
        return self._state()
    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)
    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            executed = min(int(requested), int(self.cash // (price * (1.0 + TRANSACTION_FEE))))
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed
    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping: reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value)); self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {"raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return),
                "var_target": float(var_target), "drawdown": float(drawdown),
                "executed_action": int(executed), "portfolio_value": float(portfolio_value)}
        return self._state(), float(reward), done, info

@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, states: np.ndarray) -> np.ndarray:
        return (np.asarray(states, dtype=np.float32) - self.mean) / self.std

def calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    rng = np.random.default_rng(SCALER_SEED); states = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False); state, done = env.reset(), False
        while not done:
            states.append(state.copy()); state, _, done, _ = env.step(int(rng.choice(ACTION_VALUES)))
    return np.asarray(states, dtype=np.float32)

train_hpg, test_hpg = load_hpg_bad()
calibration = calibration_states(train_hpg)
frozen_mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenScaler(frozen_mean.copy(), frozen_std.copy())
FROZEN_SCALER.mean.setflags(write=False); FROZEN_SCALER.std.setflags(write=False)
print({"train": (str(train_hpg.time.min().date()), str(train_hpg.time.max().date()), len(train_hpg)),
       "test": (str(test_hpg.time.min().date()), str(test_hpg.time.max().date()), len(test_hpg)),
       "output_dir": str(OUTPUT_DIR), "scaler_fit": "train only"})

In [ ]:
# Ã” CODE 3 â€” Q-Network, EpistemicVAE vÃ  Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class EpistemicVAE(nn.Module):
    """Joint VAE: 7 state + 11 action one-hot â†’ latent 16 â†’ reconstruction 18."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        reconstructed = self.decoder(mu + torch.randn_like(std) * std)
        return reconstructed, mu, logvar

    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        """Novelty vá»›i KLD sum/mean tÃ¹y ablation vÃ  probe zâº=Î¼+1.96Ïƒ."""
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl_terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum":
            kl = torch.sum(kl_terms, dim=-1)
        elif kl_reduction == "mean":
            kl = torch.mean(kl_terms, dim=-1)
        else:
            raise ValueError("kl_reduction pháº£i lÃ  'sum' hoáº·c 'mean'.")
        std = torch.exp(0.5 * logvar)
        reconstructed_95 = self.decoder(mu + 1.96 * std)
        reconstruction_error = torch.norm(x - reconstructed_95, p=2, dim=-1)
        return kl + reconstruction_error


class CostNetwork(nn.Module):
    """Æ¯á»›c lÆ°á»£ng VaR khÃ´ng Ã¢m tá»« concat(s_scaled, action_onehot) âˆˆ R^18."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)


def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE(), "\n", CostNetwork())


In [ ]:
# BLOCK 4 â€” Loss, replay vÃ  Confidence-Weighted Risk Penalty (tham sá»‘ hÃ³a cho grid)
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)

def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl, recon, kl

def cost_loss(predicted_var, var_target):
    return F.huber_loss(predicted_var, var_target)

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = int(capacity); self.states = []; self.actions = []; self.var_targets = []
    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32)); self.actions.append(int(action)); self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]; del self.actions[:overflow]; del self.var_targets[:overflow]
    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0: raise RuntimeError("KhÃ´ng thá»ƒ sample replay buffer rá»—ng.")
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return (np.asarray([self.states[i] for i in indices]), np.asarray([self.actions[i] for i in indices]),
                np.asarray([self.var_targets[i] for i in indices], dtype=np.float32))
    def __len__(self): return len(self.states)

def action_scores(q_network, vae, cost_network, state, beta, config, collect_details=False):
    scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1); action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval(); cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        predicted_var = cost_network(state_batch, action_batch)
        confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
        penalty = torch.where(predicted_var < ZETA, torch.zeros_like(predicted_var), confidence * predicted_var)
        scores = q_values - penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None if not collect_details else {
        "novelty": novelty_raw.detach().cpu().numpy(), "predicted_var": predicted_var.detach().cpu().numpy(),
        "penalty": penalty.detach().cpu().numpy(),
    }
    return int(ACTION_VALUES[action_index]), action_index, details

def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config):
    raw_states, action_indices, var_targets = replay.sample(config["vae_batch_size"])
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE); actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions); target = torch.cat([states, actions], dim=-1)
    loss_vae, recon, kl = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True); loss_vae.backward(); torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0); vae_optimizer.step()
    targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
    predicted = cost_network(states, actions); loss_c = cost_loss(predicted, targets)
    cost_optimizer.zero_grad(set_to_none=True); loss_c.backward(); torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0); cost_optimizer.step()
    return float(loss_vae.detach().cpu()), float(recon.detach().cpu()), float(kl.detach().cpu()), float(loss_c.detach().cpu())

def random_bootstrap(replay: ReplayBuffer, config):
    rng = np.random.default_rng(CURRENT_RUN_SEED)
    for _ in range(int(config["bootstrap_trajectories"])):
        env = TradingEnv(train_hpg, reward_shaping=False); state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11)); states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index])); var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)

def collect_episode(env, q_network, vae, cost_network, beta, config):
    state, done = env.reset(), False; states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy()); rewards.append(reward)
        actions.append(action_index); dones.append(done); var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def q_update(q_network, optimizer, trajectory, robust: bool):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE); an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE); losses = []; q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return losses

In [ ]:
# CODE 5 — Huấn luyện một cấu hình trên 20 seed, checkpoint độc lập theo tài khoản
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)

def evaluate_policy(q_network, vae, cost_network, beta, config, data, previous_row=None):
    env = TradingEnv(evaluation_frame(data, previous_row), reward_shaping=config["reward_shaping"])
    state, done, novelty_values = env.reset(), False, []
    while not done:
        action, _, details = action_scores(q_network, vae, cost_network, state, beta, config, collect_details=True)
        novelty_values.extend(details["novelty"].tolist())
        state, _, done, _ = env.step(action)
    return (np.asarray(env.portfolio_history, dtype=np.float64),
            np.asarray(env.var_targets, dtype=np.float64),
            np.asarray(novelty_values, dtype=np.float64))

def period_metrics(portfolio: np.ndarray, dates: Sequence[pd.Timestamp], var_targets=None) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0])
    roi = float(profit / max(abs(portfolio[0]), 1e-8) * 100.0)
    dates = pd.Series(dates).reset_index(drop=True)
    days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = float(portfolio[-1] / max(portfolio[0], 1e-8))
    arr = float((ratio ** (365.25 / days) - 1.0) * 100.0) if ratio > 0 else -100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else float((annual_return - RISK_FREE_RATE_PERCENT) / volatility)
    peaks = np.maximum.accumulate(portfolio)
    mdd = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    violations = int(np.sum(np.asarray(var_targets) > ZETA)) if var_targets is not None else 0
    return {"profit": profit, "roi": roi, "arr": arr, "sharpe": sharpe,
            "max_drawdown": mdd, "violations": violations}

def run_one_seed(seed: int, progress_bar=None):
    set_seed(seed); config = dict(RUN_CONFIG)
    q_network = QNetwork().to(DEVICE)
    vae = EpistemicVAE(latent_dim=int(config["vae_latent_dim"])).to(DEVICE)
    cost_network = CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=float(config["vae_lr"]))
    cost_optimizer = torch.optim.Adam(cost_network.parameters(), lr=COST_LR)
    replay = ReplayBuffer(config["vae_replay_capacity"])
    q_losses, vae_losses, recon_losses, kl_losses, cost_losses, curve_rows = [], [], [], [], [], []

    random_bootstrap(replay, config)
    if progress_bar is not None: progress_bar.set_postfix(seed=seed, phase="aux-warmup")
    for _ in range(int(config["bootstrap_updates"])):
        lv, lr, lk, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
        vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)

    for episode in range(EPISODES):
        beta = max(float(config["beta_min"]), float(config["beta_0"]) * float(config["beta_decay"]) ** episode)
        trajectory = collect_episode(TradingEnv(train_hpg, reward_shaping=True),
                                     q_network, vae, cost_network, beta, config)
        replay.add(trajectory[0], trajectory[3], trajectory[6])
        q_losses.extend(q_update(q_network, q_optimizer, trajectory, True))
        for _ in range(int(config["online_aux_updates"])):
            lv, lr, lk, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
            vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)

        train_portfolio, train_var, _ = evaluate_policy(q_network, vae, cost_network, beta, config, train_hpg)
        test_portfolio, test_var, _ = evaluate_policy(
            q_network, vae, cost_network, beta, config, test_hpg, train_hpg.iloc[-1])
        train_metrics = period_metrics(train_portfolio, train_hpg["time"], train_var)
        test_metrics = period_metrics(test_portfolio, test_hpg["time"], test_var)
        for split, values in (("train", train_metrics), ("test", test_metrics)):
            curve_rows.append({"config_id": config["config_id"], "seed": int(seed),
                               "episode": episode + 1, "split": split, "beta": float(beta), **values})
        if progress_bar is not None and ((episode + 1) % 5 == 0 or episode + 1 == EPISODES):
            progress_bar.set_postfix(seed=seed, episode=f"{episode + 1}/{EPISODES}",
                                     test_sharpe=f"{test_metrics['sharpe']:.3f}")

    train_portfolio, train_var, _ = evaluate_policy(q_network, vae, cost_network, beta, config, train_hpg)
    test_portfolio, test_var, test_novelty = evaluate_policy(
        q_network, vae, cost_network, beta, config, test_hpg, train_hpg.iloc[-1])
    train_metrics = period_metrics(train_portfolio, train_hpg["time"], train_var)
    test_metrics = period_metrics(test_portfolio, test_hpg["time"], test_var)

    metric_row = {"config_slot": CONFIG_SLOT, "config_id": config["config_id"], "seed": int(seed),
                  "episodes": EPISODES, "gamma": GAMMA, "beta_0": config["beta_0"],
                  "beta_decay": config["beta_decay"], "beta_min": config["beta_min"], **LOCKED_VAE_CONFIG}
    metric_row.update({f"train_{key}": value for key, value in train_metrics.items()})
    metric_row.update({f"test_{key}": value for key, value in test_metrics.items()})
    metric_row.update({
        "gap_profit": train_metrics["profit"] - test_metrics["profit"],
        "gap_roi": train_metrics["roi"] - test_metrics["roi"],
        "gap_arr": train_metrics["arr"] - test_metrics["arr"],
        "gap_sharpe": train_metrics["sharpe"] - test_metrics["sharpe"],
        "q_loss_mean": float(np.mean(q_losses)), "vae_loss_mean": float(np.mean(vae_losses)),
        "vae_loss_last": float(vae_losses[-1]), "vae_recon_last": float(recon_losses[-1]),
        "vae_kl_last": float(kl_losses[-1]), "cost_loss_mean": float(np.mean(cost_losses)),
        "cost_loss_last": float(cost_losses[-1]), "novelty_mean": float(np.mean(test_novelty)),
        "novelty_std": float(np.std(test_novelty)), "novelty_max": float(np.max(test_novelty)),
    })
    numeric_values = [value for value in metric_row.values() if isinstance(value, (int, float))]
    metric_row["finite"] = bool(np.isfinite(np.asarray(numeric_values, dtype=float)).all())
    loss_rows = []
    for loss_name, values in (("Q loss", q_losses), ("VAE loss", vae_losses), ("Cost loss", cost_losses)):
        for update, value in enumerate(values):
            loss_rows.append({"config_id": config["config_id"], "seed": int(seed),
                              "loss_name": loss_name, "update": update, "loss": float(value)})
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metric_row, curve_rows, loss_rows

METRICS_CSV = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_20seed_metrics.csv"
CURVES_CSV = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_episode_curves.csv"
LOSSES_CSV = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_losses.csv"
ERRORS_CSV = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_errors.csv"

def append_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)

def completed_seeds() -> set[int]:
    if not RESUME or not METRICS_CSV.exists(): return set()
    frame = pd.read_csv(METRICS_CSV)
    expected = {"config_slot": CONFIG_SLOT, "config_id": SELECTED_UCB["config_id"],
                "beta_0": SELECTED_UCB["beta_0"], "beta_decay": SELECTED_UCB["beta_decay"],
                "beta_min": SELECTED_UCB["beta_min"],
                "vae_latent_dim": LOCKED_VAE_CONFIG["vae_latent_dim"],
                "vae_lr": LOCKED_VAE_CONFIG["vae_lr"],
                "vae_beta_kl": LOCKED_VAE_CONFIG["vae_beta_kl"]}
    for column, value in expected.items():
        if column not in frame.columns: raise ValueError(f"CSV resume thiếu cột {column}: {METRICS_CSV}")
        observed = frame[column].dropna().unique()
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            if any(not np.isclose(float(item), float(value)) for item in observed):
                raise ValueError(f"CSV resume không khớp {column}={value}: {observed}")
        elif any(str(item) != str(value) for item in observed):
            raise ValueError(f"CSV resume không khớp {column}={value}: {observed}")
    return set(frame["seed"].astype(int)).intersection(VALIDATION_SEEDS)

done = completed_seeds()
if RUN_TRAINING:
    progress = tqdm(VALIDATION_SEEDS, total=len(VALIDATION_SEEDS),
                    desc=f"{SELECTED_UCB['config_id']} — 20 independent seeds",
                    unit="seed", dynamic_ncols=True, leave=True, mininterval=1.0)
    for seed in progress:
        if seed in done:
            progress.set_postfix(seed=seed, status="checkpoint-skip"); continue
        started = time.perf_counter()
        try:
            metric_row, curve_rows, loss_rows = run_one_seed(seed, progress_bar=progress)
            metric_row["elapsed_seconds"] = float(time.perf_counter() - started)
            append_frame(CURVES_CSV, pd.DataFrame(curve_rows))
            append_frame(LOSSES_CSV, pd.DataFrame(loss_rows))
            # Metrics ghi cuối cùng và là completion marker cho resume.
            append_frame(METRICS_CSV, pd.DataFrame([metric_row]))
            progress.set_postfix(seed=seed, test_profit=f"{metric_row['test_profit']:.2f}",
                                 test_sharpe=f"{metric_row['test_sharpe']:.3f}")
        except Exception as error:
            append_frame(ERRORS_CSV, pd.DataFrame([{"config_id": SELECTED_UCB["config_id"],
                "seed": int(seed), "error_type": type(error).__name__, "error": str(error),
                "traceback": traceback.format_exc()}]))
            print(f"FAILED seed={seed}: {type(error).__name__}: {error}", flush=True)
        finally:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Đã hoàn tất vòng chạy. Metrics:", METRICS_CSV)
else:
    print("RUN_TRAINING=False — bỏ qua huấn luyện và chỉ đọc checkpoint hiện có.")

In [ ]:
# CODE 6 — Báo cáo độc lập và biểu đồ chẩn đoán overfit cho cấu hình hiện tại
if not METRICS_CSV.exists():
    raise RuntimeError(f"Chưa có kết quả: {METRICS_CSV}")

metrics = (pd.read_csv(METRICS_CSV)
           .drop_duplicates(["config_id", "seed"], keep="last")
           .sort_values("seed").reset_index(drop=True))
curves = (pd.read_csv(CURVES_CSV)
          .drop_duplicates(["config_id", "seed", "episode", "split"], keep="last")
          if CURVES_CSV.exists() else pd.DataFrame())
losses = (pd.read_csv(LOSSES_CSV)
          .drop_duplicates(["config_id", "seed", "loss_name", "update"], keep="last")
          if LOSSES_CSV.exists() else pd.DataFrame())

if set(metrics["config_id"].astype(str)) != {SELECTED_UCB["config_id"]}:
    raise ValueError("Metrics chứa config khác với cấu hình được chọn trên tài khoản này.")
if not metrics["finite"].astype(bool).all(): print("WARNING: Có seed chứa metric không hữu hạn.")

summary = {"config_slot": CONFIG_SLOT, "config_id": SELECTED_UCB["config_id"],
           "beta_0": SELECTED_UCB["beta_0"], "beta_decay": SELECTED_UCB["beta_decay"],
           "beta_min": SELECTED_UCB["beta_min"], "seeds_completed": int(metrics["seed"].nunique())}
for split in ("train", "test"):
    for metric in ("profit", "roi", "arr", "sharpe", "max_drawdown", "violations"):
        column = f"{split}_{metric}"
        summary[f"{column}_mean"] = float(metrics[column].mean())
        summary[f"{column}_std"] = float(metrics[column].std(ddof=1))
for metric in ("profit", "roi", "arr", "sharpe"):
    column = f"gap_{metric}"
    summary[f"{column}_mean"] = float(metrics[column].mean())
    summary[f"{column}_std"] = float(metrics[column].std(ddof=1))

summary_frame = pd.DataFrame([summary])
SUMMARY_CSV = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_20seed_summary.csv"
SUMMARY_JSON = OUTPUT_DIR / f"{SELECTED_UCB['config_id']}_20seed_report.json"
summary_frame.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps({"run_config": RUN_CONFIG, "seed_range": list(VALIDATION_SEEDS),
                                    "summary": summary}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"### {SELECTED_UCB['config_id']} — kết quả độc lập trên tài khoản hiện tại")
display_columns = ["config_id", "seed", "train_roi", "test_roi", "gap_roi",
                   "train_arr", "test_arr", "gap_arr", "train_sharpe", "test_sharpe",
                   "gap_sharpe", "test_max_drawdown", "test_violations", "finite"]
try:
    print(metrics[display_columns].to_markdown(index=False, floatfmt=".6f"))
    print("\n### Mean ± Std"); print(summary_frame.to_markdown(index=False, floatfmt=".6f"))
except ImportError:
    print(metrics[display_columns].to_string(index=False)); print(summary_frame.to_string(index=False))

PLOT_PREFIX = OUTPUT_DIR / SELECTED_UCB["config_id"]

# 1) Metric test cuối kỳ mean ± std, cùng phong cách Battle of 3 Variants.
items = [("test_profit", "Final Profit"), ("test_roi", "ROI (%)"), ("test_arr", "ARR (%)"),
         ("test_sharpe", "Sharpe Ratio"), ("test_max_drawdown", "Max Drawdown (%)"),
         ("test_violations", "Constraint Violations")]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (column, title) in zip(axes.ravel(), items):
    std_value = metrics[column].std(ddof=1)
    std_value = float(std_value) if np.isfinite(std_value) else 0.0
    ax.bar([SELECTED_UCB["config_id"]], [metrics[column].mean()],
           yerr=[std_value], capsize=5)
    ax.set_title(f"Test — {title} (mean ± std)"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
FINAL_METRICS_PNG = Path(f"{PLOT_PREFIX}_final_test_metrics_mean_std.png")
fig.savefig(FINAL_METRICS_PNG, dpi=220, bbox_inches="tight"); plt.show()

# 2) Train–test theo episode; dải màu là ±1 std trên các seed.
GENERALIZATION_PNG = Path(f"{PLOT_PREFIX}_generalization_gap_by_episode.png")
if not curves.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    for ax, metric, title in zip(axes.ravel(), ("profit", "roi", "arr", "sharpe"),
                                 ("Profit", "ROI (%)", "ARR (%)", "Sharpe Ratio")):
        for split, style in (("train", "-"), ("test", "--")):
            grouped = (curves[curves["split"] == split].groupby("episode")[metric]
                       .agg(["mean", "std"]).reset_index())
            x, y = grouped["episode"].to_numpy(), grouped["mean"].to_numpy()
            spread = grouped["std"].fillna(0.0).to_numpy()
            ax.plot(x, y, linestyle=style, label=f"{split} mean")
            ax.fill_between(x, y - spread, y + spread, alpha=0.14)
        ax.set(title=f"Generalization Gap — {title}", xlabel="Episode", ylabel=title)
        ax.grid(alpha=0.25); ax.legend()
    plt.tight_layout(); fig.savefig(GENERALIZATION_PNG, dpi=220, bbox_inches="tight"); plt.show()

# 3) Phân phối kết quả test qua các seed đã hoàn thành.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, column, title in zip(axes, ("test_roi", "test_arr", "test_sharpe"),
                             ("Test ROI (%)", "Test ARR (%)", "Test Sharpe")):
    ax.boxplot([metrics[column].dropna().to_numpy()],
               tick_labels=[SELECTED_UCB["config_id"]], showmeans=True)
    ax.set_title(f"20-seed distribution — {title}"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
DISTRIBUTION_PNG = Path(f"{PLOT_PREFIX}_test_distributions.png")
fig.savefig(DISTRIBUTION_PNG, dpi=220, bbox_inches="tight"); plt.show()

# 4) Gap train - test. Gap dương lớn là dấu hiệu train tốt hơn test.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, column, title in zip(axes, ("gap_roi", "gap_arr", "gap_sharpe"),
                             ("ROI gap", "ARR gap", "Sharpe gap")):
    ax.boxplot([metrics[column].dropna().to_numpy()],
               tick_labels=[SELECTED_UCB["config_id"]], showmeans=True)
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Train − Test — {title}"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
GAP_DISTRIBUTION_PNG = Path(f"{PLOT_PREFIX}_overfit_gap_distributions.png")
fig.savefig(GAP_DISTRIBUTION_PNG, dpi=220, bbox_inches="tight"); plt.show()

# 5) Scatter train–test theo seed; đường chéo biểu diễn train = test.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, title in zip(axes, ("roi", "arr", "sharpe"),
                             ("ROI (%)", "ARR (%)", "Sharpe Ratio")):
    x, y = metrics[f"train_{metric}"].to_numpy(), metrics[f"test_{metric}"].to_numpy()
    ax.scatter(x, y, c=metrics["seed"], cmap="viridis")
    low, high = float(min(x.min(), y.min())), float(max(x.max(), y.max()))
    ax.plot([low, high], [low, high], "--", color="gray")
    for seed, xv, yv in zip(metrics["seed"], x, y):
        ax.annotate(str(int(seed)), (xv, yv), fontsize=7, alpha=0.75)
    ax.set(title=f"Train vs Test — {title}", xlabel=f"Train {title}", ylabel=f"Test {title}")
    ax.grid(alpha=0.25)
plt.tight_layout()
TRAIN_TEST_SCATTER_PNG = Path(f"{PLOT_PREFIX}_train_test_scatter.png")
fig.savefig(TRAIN_TEST_SCATTER_PNG, dpi=220, bbox_inches="tight"); plt.show()

# 6) Loss convergence trung bình trên các seed.
LOSSES_PNG = Path(f"{PLOT_PREFIX}_loss_convergence.png")
if not losses.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, loss_name in zip(axes, ("Q loss", "VAE loss", "Cost loss")):
        grouped = (losses[losses["loss_name"] == loss_name].groupby("update")["loss"]
                   .agg(["mean", "std"]).reset_index())
        if not grouped.empty:
            ax.plot(grouped["update"], np.maximum(grouped["mean"], 1e-12), label=loss_name)
            ax.set_yscale("log")
        ax.set(title=loss_name, xlabel="Update", ylabel="Loss"); ax.grid(alpha=0.25)
    plt.tight_layout(); fig.savefig(LOSSES_PNG, dpi=220, bbox_inches="tight"); plt.show()

completed_count = int(metrics["seed"].nunique())
print({"config": SELECTED_UCB, "seeds_completed": completed_count,
       "expected_seeds": len(VALIDATION_SEEDS),
       "missing_seeds": sorted(set(VALIDATION_SEEDS) - set(metrics["seed"].astype(int))),
       "mean_gap_roi": summary["gap_roi_mean"], "mean_gap_arr": summary["gap_arr_mean"],
       "mean_gap_sharpe": summary["gap_sharpe_mean"],
       "overfit_hint": "Gap train-test dương lớn và ổn định qua seed là dấu hiệu cần kiểm tra overfit.",
       "warning": "Chỉ kết luận khi tài khoản hiện tại hoàn thành đủ 20 seed."})

saved_outputs = [METRICS_CSV, CURVES_CSV, LOSSES_CSV, SUMMARY_CSV, SUMMARY_JSON,
                 FINAL_METRICS_PNG, DISTRIBUTION_PNG, GAP_DISTRIBUTION_PNG,
                 TRAIN_TEST_SCATTER_PNG]
if not curves.empty: saved_outputs.append(GENERALIZATION_PNG)
if not losses.empty: saved_outputs.append(LOSSES_PNG)
print("\n=== FILE KẾT QUẢ CỦA RIÊNG TÀI KHOẢN NÀY ===")
for path in saved_outputs:
    if path.exists(): print("-", path)